# TUGAS PRAKTIKUM PERTEMUAN 9 — LATIHAN MANDIRI

**Nama:** Habil Arrafi  
**NIM:** 2411511014  
**Mata Kuliah:** Pemrosesan Paralel dengan Python

Notebook ini hanya berisi pengerjaan bagian **Latihan Mandiri** sesuai instruksi modul.

## 1. Dining Philosophers dengan N Berbeda

Pada percobaan ini digunakan solusi arbitrator dengan `Semaphore(N-1)`. Semaphore membatasi jumlah filsuf yang boleh mencoba mengambil garpu secara bersamaan, sehingga tidak semua filsuf dapat memegang satu garpu dan saling menunggu. Throughput dihitung dari total aktivitas makan dibagi waktu eksekusi.

In [ ]:
import threading
import time

def uji_dining_philosophers(N, putaran=100):
    garpu = [threading.Lock() for _ in range(N)]
    arbitrator = threading.Semaphore(N - 1)
    jumlah_makan = [0] * N

    def filsuf(i):
        kiri = garpu[i]
        kanan = garpu[(i + 1) % N]

        for _ in range(putaran):
            with arbitrator:
                with kiri:
                    with kanan:
                        jumlah_makan[i] += 1
                        time.sleep(0.001)

    threads = [
        threading.Thread(target=filsuf, args=(i,))
        for i in range(N)
    ]

    mulai = time.perf_counter()

    for thread in threads:
        thread.start()

    for thread in threads:
        thread.join()

    waktu = time.perf_counter() - mulai
    total_makan = sum(jumlah_makan)
    throughput = total_makan / waktu

    return total_makan, waktu, throughput, jumlah_makan

hasil_dining = {}

for N in [3, 7, 10]:
    total, waktu, throughput, detail = uji_dining_philosophers(N)
    hasil_dining[N] = throughput

    print(f"Jumlah filsuf       : {N}")
    print(f"Total makan         : {total}")
    print(f"Waktu eksekusi      : {waktu:.3f} detik")
    print(f"Throughput          : {throughput:.2f} makan/detik")
    print(f"Makan tiap filsuf   : {detail}")
    print("-" * 40)

**Analisis:**

Berdasarkan hasil pengujian, throughput tidak selalu meningkat secara linear terhadap jumlah filsuf. Penambahan jumlah filsuf memang meningkatkan jumlah aktivitas yang dapat dilakukan, tetapi setiap filsuf tetap membutuhkan dua garpu. Karena jumlah garpu terbatas, beberapa filsuf harus menunggu sampai garpu tersedia. Selain itu, penggunaan lock dan pergantian eksekusi antar-thread juga menimbulkan overhead. Oleh karena itu, penambahan jumlah filsuf tidak selalu sebanding dengan peningkatan throughput.

## 2. Readers-Writers dengan Prioritas Writer

Pada program ini writer diberikan prioritas. Ketika writer datang, `writer_waiting` akan bertambah sehingga reader baru tidak dapat mulai membaca. Reader yang sudah aktif tetap diperbolehkan menyelesaikan prosesnya. Setelah seluruh reader selesai, writer dapat mengakses data secara eksklusif.

In [ ]:
import threading
import time

data = {"nilai": 0}
read_count = 0
writer_waiting = 0

mutex = threading.Lock()
condition = threading.Condition(mutex)

log = []

def reader(id_reader):
    global read_count

    with condition:
        while writer_waiting > 0:
            condition.wait()
        read_count += 1
        nilai_dibaca = data["nilai"]

    log.append(f"Reader-{id_reader} membaca nilai {nilai_dibaca}")
    time.sleep(0.1)

    with condition:
        read_count -= 1
        if read_count == 0:
            condition.notify_all()

def writer(id_writer, nilai_baru):
    global writer_waiting

    with condition:
        writer_waiting += 1
        while read_count > 0:
            condition.wait()

        log.append(f"Writer-{id_writer} mulai menulis")
        data["nilai"] = nilai_baru
        time.sleep(0.1)
        log.append(f"Writer-{id_writer} selesai menulis nilai {nilai_baru}")

        writer_waiting -= 1
        condition.notify_all()

# Reader pertama mulai membaca lebih dahulu.
r1 = threading.Thread(target=reader, args=(1,))
r2 = threading.Thread(target=reader, args=(2,))

r1.start()
r2.start()
time.sleep(0.02)

# Writer datang ketika reader sedang membaca.
w1 = threading.Thread(target=writer, args=(1, 100))
w1.start()
time.sleep(0.02)

# Reader baru harus menunggu writer selesai.
r3 = threading.Thread(target=reader, args=(3,))
r3.start()

for thread in [r1, r2, w1, r3]:
    thread.join()

for pesan in log:
    print(pesan)

print("\nNilai akhir data:", data["nilai"])

**Analisis:**

Prioritas writer diterapkan dengan variabel `writer_waiting`. Ketika ada writer yang menunggu, reader baru harus menunggu melalui `condition.wait()`. Reader yang sudah membaca tidak dihentikan secara paksa dan tetap menyelesaikan prosesnya. Setelah reader aktif selesai, writer memperoleh akses eksklusif. Namun, jika writer terus berdatangan, reader baru dapat mengalami starvation karena harus terus menunggu giliran writer.

## 3. Sleeping Barber dengan Dua Tukang Cukur

Program berikut menggunakan satu antrean bersama dengan kapasitas tiga pelanggan. Dua thread tukang cukur mengambil pelanggan dari antrean tersebut. Tukang cukur yang lebih dahulu tersedia akan mengambil pelanggan berikutnya. Jika antrean penuh, pelanggan yang datang akan ditolak.

In [ ]:
import threading
import queue
import time

KAPASITAS_TUNGGU = 3
JUMLAH_PELANGGAN = 10

antrean = queue.Queue(maxsize=KAPASITAS_TUNGGU)
STOP = None

hasil_layanan = []
pelanggan_ditolak = []
lock_hasil = threading.Lock()

def tukang_cukur(id_cukur):
    while True:
        pelanggan = antrean.get()

        try:
            if pelanggan is STOP:
                break

            print(f"Tukang cukur-{id_cukur} melayani pelanggan-{pelanggan}")
            time.sleep(0.15)

            with lock_hasil:
                hasil_layanan.append((pelanggan, id_cukur))

            print(f"Tukang cukur-{id_cukur} selesai melayani pelanggan-{pelanggan}")

        finally:
            antrean.task_done()

# Menjalankan dua tukang cukur.
barber_threads = [
    threading.Thread(target=tukang_cukur, args=(1,)),
    threading.Thread(target=tukang_cukur, args=(2,))
]

for thread in barber_threads:
    thread.start()

# Simulasi pelanggan datang ke tempat cukur.
for nomor in range(1, JUMLAH_PELANGGAN + 1):
    try:
        antrean.put_nowait(nomor)
        print(f"Pelanggan-{nomor} masuk ruang tunggu")
    except queue.Full:
        pelanggan_ditolak.append(nomor)
        print(f"Pelanggan-{nomor} pergi karena ruang tunggu penuh")

    time.sleep(0.02)

# Menunggu seluruh pelanggan dalam antrean selesai dilayani.
antrean.join()

# Mengirim tanda berhenti kepada kedua tukang cukur.
for _ in barber_threads:
    antrean.put(STOP)

for thread in barber_threads:
    thread.join()

print("\nJumlah pelanggan dilayani:", len(hasil_layanan))
print("Jumlah pelanggan ditolak:", len(pelanggan_ditolak))
print("Daftar hasil layanan:", sorted(hasil_layanan))

**Analisis:**

Dua tukang cukur menggunakan antrean yang sama sehingga pelanggan berikutnya akan dilayani oleh tukang cukur yang lebih dahulu tersedia. Penggunaan `Queue` membantu mengatur pembagian pelanggan dengan aman ketika diakses oleh beberapa thread. Kapasitas ruang tunggu dibatasi sebanyak tiga pelanggan. Jika antrean penuh, pelanggan baru akan pergi. Dengan adanya dua tukang cukur, pelayanan dapat berlangsung lebih cepat karena dua pelanggan bisa dilayani secara bersamaan.

## 4. Refleksi Singkat

Menurut saya, masalah **Producer-Consumer** paling mirip dengan masalah nyata dalam pengembangan software, khususnya pada web server. Ketika banyak pengguna mengirim permintaan secara bersamaan, permintaan tersebut dapat dimasukkan ke dalam antrean untuk diproses oleh beberapa worker. Producer berperan menghasilkan permintaan, sedangkan consumer mengambil dan memproses permintaan tersebut. Antrean membantu mengatur pekerjaan agar tidak saling bertabrakan. Dengan cara ini, server dapat mengelola banyak permintaan secara lebih teratur dan penggunaan resource menjadi lebih terkendali.

## Kesimpulan

1. Solusi arbitrator pada Dining Philosophers mencegah deadlock dengan membatasi jumlah filsuf yang mengambil garpu secara bersamaan.
2. Prioritas writer membuat writer yang sedang menunggu dapat masuk sebelum reader baru, tetapi reader dapat mengalami starvation.
3. Dua tukang cukur dapat melayani pelanggan dari satu antrean bersama, dengan kapasitas ruang tunggu yang terbatas.